In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set random seed for reproducibility
np.random.seed(42)
num_users = 10000

# 1. Generate Frontend PDP Views
user_ids = [f"USR_{i:06d}" for i in range(1, num_users + 1)]
# Randomly assign 50% to Control (Standard Delivery Info) and 50% to Variant (Guaranteed Countdown Timer)
variants = np.random.choice(['Control', 'Variant'], size=num_users, p=[0.5, 0.5])

pdp_data = {
    'user_id': user_ids,
    'variant_group': variants,
    'timestamp': [datetime(2026, 5, 1) + timedelta(days=float(np.random.uniform(0, 7))) for _ in range(num_users)]
}
df_pdp = pd.DataFrame(pdp_data)

# 2. Generate Orders (Injecting a slight conversion lift into the Variant group)
orders = []
order_counter = 1

for index, row in df_pdp.iterrows():
    # Base conversion is 10% for Control, Variant gets a boost to 12.5%
    conv_probability = 0.125 if row['variant_group'] == 'Variant' else 0.10

    if np.random.rand() < conv_probability:
        order_time = row['timestamp'] + timedelta(minutes=float(np.random.uniform(5, 60)))
        orders.append({
            'order_id': f"ORD_{order_counter:06d}",
            'user_id': row['user_id'],
            'timestamp': order_time,
            'order_value_pln': round(np.random.exponential(scale=150) + 30, 2) # realistic retail basket sizes
        })
        order_counter += 1

df_orders = pd.DataFrame(orders)

# 3. Generate Backend Logistics Performance (Injecting SLA breaches due to higher variant volume)
logistics = []
regions = ['Warsaw_Hub', 'Poznan_Hub', 'Rural_East', 'Southern_Nodes']

for index, row in df_orders.iterrows():
    # Fetch user variant to simulate operational stress from the variant group
    user_variant = df_pdp[df_pdp['user_id'] == row['user_id']]['variant_group'].values[0]
    region = np.random.choice(regions, p=[0.4, 0.3, 0.15, 0.15])

    promised_days = 1 if region in ['Warsaw_Hub', 'Poznan_Hub'] else 3
    promised_date = row['timestamp'].date() + timedelta(days=promised_days)

    # In rural areas, the variant's higher demand causes logistics bottlenecks (SLA breaches)
    if user_variant == 'Variant' and region in ['Rural_East', 'Southern_Nodes']:
        actual_days = promised_days + np.random.choice([0, 1, 2], p=[0.6, 0.3, 0.1])
    else:
        actual_days = promised_days + np.random.choice([0, 1], p=[0.95, 0.05])

    actual_date = row['timestamp'].date() + timedelta(days=int(actual_days))

    logistics.append({
        'order_id': row['order_id'],
        'region': region,
        'promised_delivery_date': promised_date,
        'actual_delivery_date': actual_date
    })

df_logistics = pd.DataFrame(logistics)

df_pdp.to_csv('pdp_views.csv', index=False)
df_orders.to_csv('orders.csv', index=False)
df_logistics.to_csv('delivery_performance.csv', index=False)

print("SUCCESS: Files generated! Look at the left folder sidebar in Colab to download your 3 CSV files.")

SUCCESS: Files generated! Look at the left folder sidebar in Colab to download your 3 CSV files.
